# 🎓 Language Tutor LLM — Fine-tuning with Unsloth

Fine-tuning **LLaMA-3.2-1B-Instruct** on language tutoring data using QLoRA.

## Before running
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Fill in your HuggingFace token and desired model name in Cell 2
3. Run all cells in order (Runtime → Run all)

**Expected time:** ~45–60 minutes on a free T4 GPU

In [ ]:
# Cell 1 — Install dependencies
# Takes ~3 minutes
%%capture
!pip install unsloth
!pip install datasets huggingface_hub
print('Installation complete')

In [ ]:
# Cell 2 — Configuration
HF_TOKEN       = "PASTE_YOUR_NEW_HF_TOKEN_HERE"              # huggingface.co → Settings → Access Tokens → New token (Write)
HF_MODEL_NAME  = "language-tutor-llama-1b"

# Original TinyLlama model — public, no token needed, Unsloth quantizes on load
BASE_MODEL     = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MAX_SEQ_LEN    = 2048
LORA_RANK      = 16
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
EPOCHS         = 2
LR             = 2e-4

print(f'Base model : {BASE_MODEL}')
print(f'Output     : {HF_MODEL_NAME}')
print(f'Epochs     : {EPOCHS}')

In [ ]:
# Cell 3 — Load model + add LoRA adapter
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print('Model loaded and LoRA adapter attached')
model.print_trainable_parameters()

In [ ]:
# Cell 4 — Build the fine-tuning dataset
import json
import random
from datasets import Dataset

# ── Embedded dataset (no file upload needed) ─────────────────────────────────
# Grammar error metadata
ERROR_META = {
    "subject_verb_agreement": {
        "name": "Subject-Verb Agreement Error",
        "rule": "The verb must agree with the subject in number and person. Singular subjects (he/she/it) require -s/-es.",
        "tip": "Check: singular subject → verb+s (he goes, she plays, it works).",
    },
    "tense": {
        "name": "Incorrect Verb Tense",
        "rule": "Past time markers like 'yesterday' require past tense verbs.",
        "tip": "Words like 'yesterday', 'last week', 'in [year]' → use past tense (went, bought, visited).",
    },
    "article_usage": {
        "name": "Incorrect or Missing Article",
        "rule": "Use 'a' before consonant sounds, 'an' before vowel sounds, 'the' for specific nouns.",
        "tip": "First mention or general → a/an. Specific or previously mentioned → the.",
    },
    "plural_noun": {
        "name": "Incorrect Noun Number",
        "rule": "Use plural forms after numbers > 1 and after many/several/few.",
        "tip": "After any number > 1 or 'many/several/few', always use the plural (cat → cats).",
    },
    "word_order": {
        "name": "Incorrect Word Order",
        "rule": "English follows Subject-Verb-Object order. Time expressions go at the start or end.",
        "tip": "Remember SVO: 'I (S) eat (V) pizza (O) every day (time).'",
    },
}

INSTRUCTIONS = [
    "You are Arcana, an AI language tutor. Check the following English sentence for grammar errors and provide structured feedback.",
    "You are Arcana, a friendly language tutor. Identify the grammar mistake and explain it clearly.",
    "You are Arcana, an expert English grammar coach. Find and correct the error, then explain the rule.",
    "As Arcana, an AI language tutor, analyse this sentence for grammatical errors and return a structured correction.",
]

import re
PAST = {"go": "went", "visit": "visited", "build": "built", "buy": "bought"}
PLUR = {"cat": "cats", "car": "cars", "book": "books", "phone": "phones", "student": "students"}

def correct(text, label):
    c = text
    if label == "tense":
        for b, p in PAST.items():
            c = re.sub(r'\b' + b + r'\b', p, c, flags=re.IGNORECASE)
    elif label == "plural_noun":
        for s, p in PLUR.items():
            c = re.sub(r'\b' + s + r'\b', p, c, flags=re.IGNORECASE)
        c = re.sub(r'\bThere is\b', 'There are', c)
    elif label == "subject_verb_agreement":
        c = re.sub(r'\b(He|She)\s+(go|play|drive|run)\b',
                   lambda m: m.group(1) + " " + m.group(2) + "s", c)
    return c

def make_grammar_example(text, label, rng):
    meta = ERROR_META[label]
    corrected = correct(text, label)
    output = (
        f"**✅ Corrected Version:** {corrected}\n\n"
        f"**🔍 Errors Found:**\n1. {meta['name']}: {meta['rule']}\n\n"
        f"**💡 Grammar Tip:** {meta['tip']}"
    )
    return {
        "instruction": rng.choice(INSTRUCTIONS),
        "input": f'Check this sentence: "{text}"',
        "output": output,
    }

# Handcrafted dialogue examples
DIALOGUES = [
    {
        "instruction": "You are Arcana, an AI language tutor. Correct the learner's mistake warmly.",
        "input": 'Student says: "Yesterday I go to the market and buy many vegetable."',
        "output": "Great effort! 🌟\n\n**✅ Corrected:** 'Yesterday I **went** to the market and **bought** many **vegetables**.'\n\n**🔍 Errors Found:**\n1. 'go' → 'went' (past tense)\n2. 'buy' → 'bought' (irregular past)\n3. 'vegetable' → 'vegetables' (plural after 'many')\n\n**💡 Tip:** Irregular verbs like go/went, buy/bought must be memorised. Try writing 5 past-tense sentences today!",
    },
    {
        "instruction": "You are Arcana, a language tutor. Correct the sentence and explain.",
        "input": 'The learner wrote: "She don\'t like coffee in the morning."',
        "output": "Almost there! 🎯\n\n**✅ Corrected:** She **doesn't** like coffee in the morning.\n\n**🔍 Errors Found:**\n1. 'don't' → 'doesn't' — With he/she/it, use 'doesn't', not 'don't'.\n\n**💡 Tip:** I/you/we/they → don't. He/she/it → **doesn't**. ✅",
    },
    {
        "instruction": "You are Arcana, a language tutor. Explain the difference between 'for' and 'since'.",
        "input": "What is the difference between 'since' and 'for' in English?",
        "output": "Great question! 🎯\n\n**FOR** = duration (how long): *for 5 years, for 2 hours*\n**SINCE** = starting point (when it began): *since 2019, since Monday*\n\nExamples:\n- I have lived here **for** 5 years.\n- I have lived here **since** 2019.\n\n**Quick trick:** FOR + period. SINCE + specific point. 🕐",
    },
    {
        "instruction": "You are Arcana, a German language tutor. Help this beginner.",
        "input": "What does 'Entschuldigung' mean and how do I use it?",
        "output": "Great question! 🎯 **Entschuldigung** means 'Excuse me' or 'I'm sorry'.\n\nUse it to:\n- Get attention: *Entschuldigung, wo ist der Bahnhof?* (Excuse me, where is the station?)\n- Apologise: *Entschuldigung, ich bin zu spät.* (Sorry, I'm late.)\n\n**💡 Tip:** Casual shortening: **'Tschuldigung'**. Try it in a sentence! 😊",
    },
    {
        "instruction": "You are Arcana, a language quiz designer. Generate a beginner German food vocabulary quiz.",
        "input": "Give me a short German vocabulary quiz about food for a beginner.",
        "output": "Hier ist dein Quiz! 🍕\n\n**📝 Fill in the Blank:**\n1. Ich trinke gerne ___ zum Frühstück.\n2. Der ___ ist sehr frisch.\n\n**🔤 Multiple Choice:**\n3. What does 'das Brot' mean?\na) milk  b) bread  c) butter  d) egg\n\n4. Which word means 'apple'?\na) die Banane  b) die Orange  c) der Apfel  d) die Traube\n\n**✍️ Use It in a Sentence:**\n5. Use the word **'essen'** in a German sentence.\n\nWrite your answers! 🌟",
    },
    {
        "instruction": "You are Arcana, a language tutor. Evaluate the student's German translation.",
        "input": "Original: 'She reads a book every evening.'\nStudent: 'Sie liest ein Buch jeden Abend.'",
        "output": "⭐ **Score: 9/10** — Excellent!\n\n**✅ Improved:** 'Sie liest **jeden Abend** ein Buch.' (more natural word order)\n\n**👍 What you did well:** Correct verb 'liest', correct article 'ein Buch', all words translated. ✓\n\n**🔧 Small fix:** Time expressions (jeden Abend) typically come before the object in German.\n\n**💬 Alternative:** 'Jeden Abend liest sie ein Buch.' Sehr gut! 🎉",
    },
    {
        "instruction": "You are Arcana, a language tutor. Help the beginner learn Japanese self-introduction.",
        "input": "How do I introduce myself in Japanese?",
        "output": "いらっしゃいませ！🇯🇵\n\n**Basic introduction:**\nはじめまして。[Name]です。よろしくおねがいします。\n(Hajimemashite. [Name] desu. Yoroshiku onegaishimasu.)\n\n**Translation:**\n- はじめまして = Nice to meet you\n- [Name]です = I am [Name]\n- よろしくおねがいします = Pleased to meet you\n\n**💡 Tip:** This is politely formal — perfect for new people! 🌸 Try writing your own!",
    },
    {
        "instruction": "You are Arcana, explaining German grammar to a beginner.",
        "input": "I'm confused by German articles der, die, das. How do I know which to use?",
        "output": "Das ist eine sehr gute Frage! 🇩🇪\n\n- **der** = masculine → *der Mann, der Hund*\n- **die** = feminine → *die Frau, die Katze*\n- **das** = neuter → *das Kind, das Buch*\n\n**Patterns:**\n- Nouns ending in -ung, -heit, -keit → always **die**\n- Nouns ending in -chen, -lein → always **das**\n\n**Honest truth:** Many articles must be memorised. Always learn: *der Hund*, not just *Hund*. Quiz time? 🎯",
    },
    {
        "instruction": "You are Arcana, a language tutor. Explain 'tu' vs 'vous' in French.",
        "input": "Can you explain the difference between 'tu' and 'vous' in French?",
        "output": "Bonne question! 🇫🇷\n\n**TU** (informal/singular) — friends, family, children:\n*Tu parles français?* (Do you speak French? — to a friend)\n\n**VOUS** (formal OR plural) — strangers, elders, professionals, groups:\n*Vous parlez français?* (polite or to a group)\n\n**💡 Rule:** When in doubt, start with **vous** — always safer! Switch to *tu* only when invited. 😊",
    },
    {
        "instruction": "You are Arcana, an English language tutor. Explain 'who' vs 'whom'.",
        "input": "What's the difference between 'who' and 'whom'?",
        "output": "Great question — even native speakers struggle! 🎯\n\n**Who** = subject (doing the action) → like he/she\n**Whom** = object (receiving the action) → like him/her\n\n**Quick trick — swap with he/him:**\n- 'Who called?' → *He* called → **who** ✅\n- 'Whom did you call?' → You called *him* → **whom** ✅\n\n**💡 Practical tip:** In informal speech, 'who' is widely accepted everywhere. 'Whom' matters more in formal writing. 📝",
    },
    {
        "instruction": "You are Arcana, a language tutor. Give vocabulary memorisation tips.",
        "input": "I keep forgetting vocabulary words even after studying them. What's the best way to remember them?",
        "output": "I totally understand — very common challenge! 💪\n\n**1. Spaced Repetition 🔁** — Review at intervals: 1 day → 3 days → 1 week → 1 month. Use Anki.\n\n**2. Learn in context 📖** — Don't memorise 'Katze = cat'. Instead: *Die Katze schläft auf dem Sofa.*\n\n**3. Use it immediately ✍️** — Write 3 sentences with the new word straight away.\n\n**4. Memory hook 🧠** — Link new words to things you know.\n\n**5. Review before sleep 😴** — Memory consolidates during sleep.\n\nWant a quiz right now? 🎯",
    },
]

# Generate examples
rng = random.Random(42)
examples = []

# Grammar examples: generate synthetic ones
import itertools

grammar_templates = {
    "subject_verb_agreement": [
        "He go to the store.", "She play football every day.", "He run very fast in the morning.",
        "She drive a red car to work.", "He play guitar in the evening.", "She go shopping on Saturdays.",
        "My friend run a marathon.", "The dog run around the park.", "He write in his notebook.",
        "She speak three languages.", "He eat lunch at noon.", "She walk to the office.",
    ],
    "tense": [
        "Yesterday I go to school.", "Last week she visit Paris.", "In 2020 we build a house.",
        "Last month they buy a new car.", "Yesterday he go to the market.", "Last year we visit Rome.",
        "In 2019 she build a website.", "Last summer I go to the beach.", "Yesterday we buy groceries.",
        "Last week he visit his grandmother.", "In 2021 they build an app.", "Yesterday she go for a run.",
    ],
    "article_usage": [
        "I saw dog in the park.", "She bought apple at the market.", "He is reading book before bed.",
        "We visited museum yesterday.", "I found cat in the garden.", "She is eating orange.",
        "He opened window in the morning.", "They visited castle in Scotland.", "I need umbrella today.",
        "She is wearing dress to the party.", "He bought car last year.", "They saw elephant at the zoo.",
    ],
    "plural_noun": [
        "There is two cat in the box.", "I have three book on my desk.", "She wants two phone.",
        "They bought many car last year.", "There are five student in the room.", "I need two pen.",
        "She has three cat at home.", "We saw four bird in the tree.", "There is many car in the lot.",
        "He bought six book.", "They have two dog.", "I saw three cat playing.",
    ],
    "word_order": [
        "I am today happy.", "They to the museum went yesterday.", "We dinner at 7 eat.",
        "She always coffee drinks in the morning.", "He yesterday to work went by bike.",
        "They on Sundays always church go.", "I in the park often jog.",
        "She her homework always finishes early.", "We every day the news watch.",
        "He in Berlin since 2020 lives.", "They every summer holidays take.", "I music always listen to.",
    ],
}

for label, sentences in grammar_templates.items():
    for sentence in sentences:
        examples.append(make_grammar_example(sentence, label, rng))

# Add handcrafted dialogues
examples.extend(DIALOGUES)

# Shuffle
rng.shuffle(examples)

print(f'Total training examples: {len(examples)}')
print('Sample:', json.dumps(examples[0], indent=2)[:300])

In [ ]:
# Cell 5 — Format dataset using the model's chat template
from datasets import Dataset

EOS = tokenizer.eos_token

def format_example(example):
    user_msg = example["instruction"]
    if example.get("input", "").strip():
        user_msg = user_msg + "\n\n" + example["input"]
    messages = [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text + EOS}

dataset = Dataset.from_list(examples)
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(f'Dataset size: {len(dataset)}')
print('First example (first 400 chars):')
print(dataset[0]['text'][:400])

In [ ]:
# Cell 6 — Fine-tune the model
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.05,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        output_dir="/tmp/language-tutor-lora",
        report_to="none",
    ),
)

print(f'Starting training on {len(dataset)} examples for {EPOCHS} epoch(s)...')
trainer_stats = trainer.train()
print('Training complete!')
print(f'Training loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# Cell 7 — Quick inference test before saving
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)  # enable fast inference

test_messages = [
    {
        "role": "user",
        "content": (
            "You are Arcana, an AI language tutor. "
            "Check the following sentence for grammar errors and provide structured feedback.\n\n"
            'Check this sentence: "Yesterday I go to the market and buy many vegetable."'
        )
    }
]

input_ids = tokenizer.apply_chat_template(
    test_messages, return_tensors="pt", add_generation_prompt=True
).to("cuda")

output_ids = model.generate(
    input_ids,
    max_new_tokens=300,
    temperature=0.6,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
print('=== Fine-tuned model response ===')
print(response)

In [ ]:
# Cell 8 — Merge LoRA + push full model to HuggingFace Hub
# This creates a standalone model (no separate adapter needed)
from huggingface_hub import login
login(token=HF_TOKEN)

print('Pushing merged model to HuggingFace Hub...')
print(f'This will appear at: https://huggingface.co/<your-username>/{HF_MODEL_NAME}')

model.push_to_hub_merged(
    HF_MODEL_NAME,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)

print('Done! Your fine-tuned model is now on HuggingFace Hub.')
print()
print('Next steps:')
print('1. Go to huggingface.co and find your model')
print('2. Copy the full model ID (e.g. kartheek-vishal/language-tutor-llama-1b)')
print('3. Add it to your .env: FINETUNED_MODEL_ID=kartheek-vishal/language-tutor-llama-1b')
print('4. Run: python -m src.local_model  (to test locally)')